In [ ]:
## RAG Application With GPT-4o

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")

# 1. Get a Data Loader


In [ ]:
from langchain_community.document_loaders import WebBaseLoader


In [ ]:
loader = WebBaseLoader("https://docs.smith.langchain.com/user_guide")
data = loader.load()
data

# 2. Convert data to Vector Database


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(data)

In [ ]:
documents

In [ ]:
if "vector" in globals():
    try:
        vector._db.close()
        print("Old ObjectBox store closed.")
    except Exception as e:
        print("No open store:", e)

In [ ]:
from langchain_objectbox.vectorstores import ObjectBox
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector = ObjectBox.from_documents(documents, embeddings, embedding_dimensions=768)
print("ObjectBox created successfully.")

# 3. Make a RAG pipeline


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import RetrievalQA


In [ ]:
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv('GROQ_API_KEY')
llm=ChatGroq(groq_api_key=groq_api_key,model="openai/gpt-oss-20b")
prompt=ChatPromptTemplate.from_template(
    """
    Answer the questions based on the provided context only.
    Please provide the most accurate response based on the question
    <context>
    {context}
        <context>
    Questions:{input}
    """
)
prompt


In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

document_chain = create_stuff_documents_chain(llm=llm,prompt=prompt)
retriever = vector.as_retriever(search_kwargs={"k": 4})
retrieval_chain = create_retrieval_chain(retriever=retriever,combine_docs_chain=document_chain)

In [ ]:
import pprint
response = retrieval_chain.invoke({"input": "What is the document about?"})
pp = pprint.PrettyPrinter(indent=5)
pp.pprint(response["answer"])

In [ ]:
question = "Explain what is LangSmith"
result = retrieval_chain.invoke({"input": question})
pp = pprint.PrettyPrinter(indent=5)
pp.pprint(result["answer"])